# De ruido blanco a vectores gaussianos correlacionados
## Tutorial con factorizaciones de Cholesky y $LDL^\top$

Este cuaderno desarrolla el ejemplo presentado en **KalmanFilters_1.pdf**, dentro del capítulo de teoría de estimación. El objetivo es entender cómo transformar muestras gaussianas independientes en vectores aleatorios con una media y una covarianza especificadas.

Al finalizar podrás:

- interpretar la media, la covarianza y la correlación de un vector aleatorio;
- justificar matemáticamente la transformación $Y=\bar y+BZ$;
- generar muestras mediante Cholesky y $LDL^\top$;
- comparar parámetros teóricos con estimaciones muestrales;
- interpretar correctamente una elipse de covarianza;
- relacionar este procedimiento con la simulación de ruido en filtros de Kalman.


## 1. Contexto probabilístico

En teoría de estimación, una perturbación o un error de medición se modela frecuentemente mediante un vector aleatorio. Para un vector bidimensional

$$
Y=\begin{bmatrix}Y_1\\Y_2\end{bmatrix},
$$

su media y su matriz de covarianza se definen como

$$
\bar y=\mathbb E[Y],
\qquad
\Sigma=\mathbb E\!\left[(Y-\bar y)(Y-\bar y)^\top\right].
$$

Para este ejemplo se desea generar

$$
Y\sim\mathcal N\!\left(
\begin{bmatrix}1\\2\end{bmatrix},
\begin{bmatrix}1&0.5\\0.5&1\end{bmatrix}
\right).
$$

Los elementos diagonales de $\Sigma$ son las varianzas. Los términos fuera de la diagonal son covarianzas. En este caso, ambas desviaciones estándar son 1 y el coeficiente de correlación es

$$
\rho_{12}=\frac{\Sigma_{12}}{\sqrt{\Sigma_{11}\Sigma_{22}}}=0.5.
$$

Por eso esperamos una nube alargada con pendiente positiva: valores grandes de $Y_1$ tienden a acompañarse de valores grandes de $Y_2$. Correlación no significa causalidad y, en general, ausencia de correlación no significa independencia. Para variables conjuntamente gaussianas, sin embargo, incorrelación sí implica independencia.


## 2. ¿Por qué funciona la transformación lineal?

Partimos de un vector gaussiano estándar

$$Z\sim\mathcal N(0,I).$$

Sus componentes son independientes, $\mathbb E[Z]=0$ y $\operatorname{Cov}(Z)=I$. Si encontramos una matriz $B$ que satisfaga

$$BB^\top=\Sigma,$$

podemos definir

$$Y=\bar y+BZ.$$

La media resultante es

$$\mathbb E[Y]=\bar y+B\mathbb E[Z]=\bar y,$$

y la covarianza es

$$
\begin{aligned}
\operatorname{Cov}(Y)
&=\mathbb E[(Y-\bar y)(Y-\bar y)^\top]\\
&=B\,\mathbb E[ZZ^\top]B^\top\\
&=BIB^\top=BB^\top=\Sigma.
\end{aligned}
$$

El problema computacional se reduce, por tanto, a construir una raíz matricial $B$ de la covarianza.

> **Convenciones importantes.** En MATLAB, `chol(Sigma)` devuelve normalmente un factor triangular superior $A$ tal que $A^\top A=\Sigma$; por eso las diapositivas utilizan $A^\top x$. En NumPy, `np.linalg.cholesky(Sigma)` devuelve un factor triangular inferior $L$ tal que $LL^\top=\Sigma$; por eso aquí se utiliza directamente $Lz$.


## 3. Preparación del entorno

El código necesita NumPy para álgebra lineal y Plotly para la gráfica interactiva. Si hace falta instalarlos, ejecuta una vez `pip install numpy plotly`. Se usa un generador con semilla fija para que el experimento sea reproducible.


In [1]:
import numpy as np
import plotly.graph_objects as go

np.set_printoptions(precision=4, suppress=True)
rng = np.random.default_rng(seed=42)


## 4. Definición y validación de los parámetros

`ybar` tiene forma `(2, 1)`. Esta forma permite sumar la media a una matriz de muestras de forma `(2, n_samples)` mediante *broadcasting*: la misma media se añade a cada columna. Cada columna será una realización del vector aleatorio $Y$.

Una matriz de covarianza válida debe ser simétrica y semidefinida positiva. Cholesky exige, además, que sea definida positiva. Lo comprobamos examinando sus autovalores.


In [2]:
# Media y covarianza deseadas
ybar = np.array([[1.0], [2.0]])
covar = np.array([[1.0, 0.5],
                  [0.5, 1.0]])
n_samples = 5_000

eigvals = np.linalg.eigvalsh(covar)
std = np.sqrt(np.diag(covar))
corr = covar / np.outer(std, std)

print('Forma de ybar:', ybar.shape)
print('Forma de covar:', covar.shape)
print('¿Es simétrica?:', np.allclose(covar, covar.T))
print('Autovalores:', eigvals)
print('Matriz de correlación:\n', corr)

assert ybar.shape == (2, 1)
assert covar.shape == (2, 2)
assert np.allclose(covar, covar.T)
assert np.all(eigvals > 0), 'Cholesky requiere una matriz definida positiva.'


Forma de ybar: (2, 1)
Forma de covar: (2, 2)
¿Es simétrica?: True
Autovalores: [0.5 1.5]
Matriz de correlación:
 [[1.  0.5]
 [0.5 1. ]]


Los autovalores teóricos son $0.5$ y $1.5$, ambos positivos. La covarianza es, por tanto, definida positiva y admite una factorización de Cholesky.

## 5. Método de Cholesky

La factorización calcula una matriz triangular inferior $L$ tal que

$$\Sigma=LL^\top.$$

Generamos después $n_{\mathrm{samples}}$ columnas independientes de $Z\sim\mathcal N(0,I_2)$ y aplicamos $Y=\bar y+LZ$.


In [3]:
# Factor triangular inferior: covar = A_chol @ A_chol.T
A_chol = np.linalg.cholesky(covar)

# Cada columna es una realización independiente de N(0, I_2)
x_chol = rng.standard_normal((2, n_samples))

# Cada columna de y_chol es una realización de N(ybar, covar)
y_chol = ybar + A_chol @ x_chol

print('Factor de Cholesky L:\n', A_chol)
print('Comprobación L @ L.T:\n', A_chol @ A_chol.T)
print('Forma de x_chol:', x_chol.shape)
print('Forma de y_chol:', y_chol.shape)


Factor de Cholesky L:
 [[1.    0.   ]
 [0.5   0.866]]
Comprobación L @ L.T:
 [[1.  0.5]
 [0.5 1. ]]
Forma de x_chol: (2, 5000)
Forma de y_chol: (2, 5000)


Para esta matriz, NumPy obtiene aproximadamente

$$
L=\begin{bmatrix}1&0\\0.5&\sqrt{0.75}\end{bmatrix}.
$$

Así, $Y_1=1+Z_1$ y $Y_2=2+0.5Z_1+\sqrt{0.75}Z_2$. El término compartido $Z_1$ introduce la covarianza positiva.

## 6. Método $LDL^\top$

Otra posibilidad es factorizar

$$\Sigma=LDL^\top,$$

donde $L$ es triangular inferior con unos en la diagonal y $D$ es diagonal. Cuando los elementos de $D$ son no negativos, una raíz válida es

$$B_{LDL}=L\sqrt D,$$

porque $B_{LDL}B_{LDL}^\top=L\sqrt D\sqrt D\,L^\top=LDL^\top=\Sigma$.

La siguiente función implementa el algoritmo sin pivoteo usado en el código original. Es adecuada para el ejemplo, pero no es una implementación numéricamente robusta para cualquier matriz.


In [4]:
def ldl_decomposition(A):
    """Factorización LDL^T elemental para una matriz simétrica.

    No utiliza pivoteo. Requiere pivotes diagonales no nulos durante
    la recursión; se incluye como implementación didáctica.
    """
    A = np.asarray(A, dtype=float)
    n = A.shape[0]
    L = np.eye(n)
    D = np.zeros_like(A)

    for j in range(n):
        D[j, j] = (
            A[j, j]
            - L[j, :j] @ D[:j, :j] @ L[j, :j].T
        )
        if np.isclose(D[j, j], 0.0):
            raise np.linalg.LinAlgError(
                'Pivote nulo: esta implementación didáctica necesita pivoteo.'
            )

        for i in range(j + 1, n):
            L[i, j] = (
                A[i, j]
                - L[i, :j] @ D[:j, :j] @ L[j, :j].T
            ) / D[j, j]

    return L, D


L_ldl, D_ldl = ldl_decomposition(covar)

if np.any(np.diag(D_ldl) < 0):
    raise ValueError('Una covarianza no puede producir una raíz real con D negativa.')

B_ldl = L_ldl @ np.sqrt(D_ldl)
x_ldl = rng.standard_normal((2, n_samples))
y_ldl = ybar + B_ldl @ x_ldl

print('L:\n', L_ldl)
print('D:\n', D_ldl)
print('Reconstrucción L @ D @ L.T:\n', L_ldl @ D_ldl @ L_ldl.T)
print('Raíz B_LDL = L @ sqrt(D):\n', B_ldl)


L:
 [[1.  0. ]
 [0.5 1. ]]
D:
 [[1.   0.  ]
 [0.   0.75]]
Reconstrucción L @ D @ L.T:
 [[1.  0.5]
 [0.5 1. ]]
Raíz B_LDL = L @ sqrt(D):
 [[1.    0.   ]
 [0.5   0.866]]


Para esta covarianza, $B_{LDL}$ coincide con el factor de Cholesky salvo por errores de redondeo. Las dos nubes no contienen los mismos puntos porque se generaron dos conjuntos independientes de números aleatorios, pero ambas deben aproximar la misma distribución.

## 7. Ajuste de la gaussiana a las muestras

El código llama "ajuste" a calcular la media y la covarianza muestrales. No se está resolviendo aquí una optimización: para muestras gaussianas, estas estadísticas son los estimadores naturales de los parámetros.

`axis=1` indica que se promedia a lo largo de las columnas, obteniendo una media por variable. `np.cov(y_chol)` interpreta cada fila como una variable y cada columna como una observación; por defecto usa el divisor $n-1$.


In [5]:
fitted_mean = np.mean(y_chol, axis=1)
fitted_cov = np.cov(y_chol)

mean_ldl = np.mean(y_ldl, axis=1)
cov_ldl = np.cov(y_ldl)

print('Media teórica:     ', ybar.ravel())
print('Media Cholesky:    ', fitted_mean)
print('Media LDL:         ', mean_ldl)
print('\nCovarianza teórica:\n', covar)
print('\nCovarianza Cholesky:\n', fitted_cov)
print('\nCovarianza LDL:\n', cov_ldl)
print('\nError Frobenius Cholesky:', np.linalg.norm(fitted_cov - covar))
print('Error Frobenius LDL:     ', np.linalg.norm(cov_ldl - covar))


Media teórica:      [1. 2.]
Media Cholesky:     [0.9801 1.9895]
Media LDL:          [0.9913 2.0384]

Covarianza teórica:
 [[1.  0.5]
 [0.5 1. ]]

Covarianza Cholesky:
 [[0.9989 0.4699]
 [0.4699 0.99  ]]

Covarianza LDL:
 [[0.9991 0.5134]
 [0.5134 1.022 ]]

Error Frobenius Cholesky: 0.04377876850184066
Error Frobenius LDL:      0.02910312117842428


Las estadísticas no serán exactamente iguales a los valores teóricos porque se dispone de un número finito de muestras. Si aumentas `n_samples`, las diferencias tienden a disminuir, aunque no lo hacen de manera estrictamente monótona en cada realización.

## 8. Construcción de la elipse de covarianza

La elipse se obtiene a partir de los autovalores y autovectores de la covarianza:

- los autovectores determinan las direcciones principales;
- las raíces cuadradas de los autovalores determinan las longitudes básicas de los semiejes;
- `n_std` multiplica ambos semiejes.

La circunferencia unitaria se transforma mediante $V\operatorname{diag}(n_{std}\sqrt{\lambda_i})$ y se traslada al centro. Sus puntos satisfacen

$$
(y-\bar y)^\top\Sigma^{-1}(y-\bar y)=n_{std}^2.
$$

> **Precisión estadística.** En dos dimensiones, `n_std=1` delimita una distancia de Mahalanobis igual a 1 y contiene aproximadamente $1-e^{-1/2}=39.35\%$ de la probabilidad. No debe confundirse con el 68.27 % del intervalo $\mu\pm\sigma$ de una gaussiana unidimensional. Para una región bidimensional del 95 %, se necesita $n_{std}=\sqrt{\chi^2_{2,0.95}}\approx2.448$.


In [6]:
def get_cov_ellipse(cov, center, n_std=1.0, num_points=200):
    """Devuelve los puntos de una elipse de distancia Mahalanobis n_std."""
    cov = np.asarray(cov, dtype=float)
    center = np.asarray(center, dtype=float).reshape(2)

    theta = np.linspace(0.0, 2.0 * np.pi, num_points)
    unit_circle = np.array([np.cos(theta), np.sin(theta)])

    # eigh es apropiado para matrices simétricas y devuelve autovalores reales
    eigvals, eigvecs = np.linalg.eigh(cov)
    if np.any(eigvals < 0):
        raise ValueError('La matriz no es semidefinida positiva.')

    axes_lengths = n_std * np.sqrt(eigvals)
    transform = eigvecs @ np.diag(axes_lengths)
    return transform @ unit_circle + center[:, None]


ellipse_1 = get_cov_ellipse(fitted_cov, fitted_mean, n_std=1.0)

# Para chi-cuadrado con 2 grados de libertad: F(q) = 1 - exp(-q/2).
# Si q = n_std^2, entonces n_std = sqrt(-2 log(1-p)).
p_95 = 0.95
radius_95 = np.sqrt(-2.0 * np.log(1.0 - p_95))
ellipse_95 = get_cov_ellipse(fitted_cov, fitted_mean, n_std=radius_95)

print(f'Radio Mahalanobis para 95 % en 2D: {radius_95:.4f}')


Radio Mahalanobis para 95 % en 2D: 2.4477


## 9. Visualización interactiva con Plotly

Se superponen las muestras obtenidas por ambos métodos y dos elipses calculadas a partir de las muestras de Cholesky. La transparencia ayuda a distinguir la densidad de puntos. La relación de aspecto se fija en 1:1 para no deformar visualmente las elipses.


In [8]:
fig = go.Figure()

fig.add_trace(go.Scattergl(
    x=y_chol[0, :],
    y=y_chol[1, :],
    mode='markers',
    marker=dict(size=4, color='royalblue', opacity=0.28),
    name='Cholesky'
))

fig.add_trace(go.Scattergl(
    x=y_ldl[0, :],
    y=y_ldl[1, :],
    mode='markers',
    marker=dict(size=4, color='crimson', opacity=0.25),
    name='LDLᵀ'
))

fig.add_trace(go.Scatter(
    x=ellipse_1[0],
    y=ellipse_1[1],
    mode='lines',
    line=dict(color='black', width=3),
    name='d_M = 1 (≈39.3 %)'
))

fig.add_trace(go.Scatter(
    x=ellipse_95[0],
    y=ellipse_95[1],
    mode='lines',
    line=dict(color='darkgreen', width=3, dash='dash'),
    name='Región 95 %'
))

fig.add_trace(go.Scatter(
    x=[fitted_mean[0]],
    y=[fitted_mean[1]],
    mode='markers',
    marker=dict(size=11, color='gold', line=dict(color='black', width=1)),
    name='Media muestral'
))

fig.update_layout(
    title='Vectores gaussianos correlacionados mediante Cholesky y LDLᵀ',
    xaxis_title='Variable 1',
    yaxis_title='Variable 2',
    template='plotly_white',
    width=900,
    height=650,
    legend=dict(x=0.01, y=0.99),
    yaxis=dict(scaleanchor='x', scaleratio=1),
)

fig.show()


### Cómo interpretar la figura

1. La nube está centrada cerca de $(1,2)$, como prescribe $\bar y$.
2. Su orientación ascendente refleja la covarianza positiva $\Sigma_{12}=0.5$.
3. Las nubes azul y roja se superponen estadísticamente porque ambos factores satisfacen $BB^\top=\Sigma$.
4. El eje principal largo de la elipse apunta aproximadamente en la dirección $(1,1)^\top$, asociada al autovalor mayor $1.5$.
5. El eje corto apunta aproximadamente en la dirección $(1,-1)^\top$, asociada al autovalor $0.5$.

## 10. Comprobación empírica de la cobertura

La distancia de Mahalanobis al cuadrado de una muestra es

$$d_M^2=(y-\bar y)^\top\Sigma^{-1}(y-\bar y).$$

Contamos qué fracción de las muestras cae dentro de cada elipse. Se usan la media y la covarianza teóricas para que la comprobación corresponda directamente al modelo especificado.


In [9]:
centered = y_chol - ybar
inv_covar = np.linalg.inv(covar)
mahalanobis_sq = np.einsum('in,ij,jn->n', centered, inv_covar, centered)

coverage_1 = np.mean(mahalanobis_sq <= 1.0)
coverage_95 = np.mean(mahalanobis_sq <= radius_95**2)

print(f'Cobertura empírica para d_M ≤ 1:   {coverage_1:.2%}')
print(f'Valor teórico correspondiente:      {1 - np.exp(-0.5):.2%}')
print(f'Cobertura empírica de la región 95 %: {coverage_95:.2%}')


Cobertura empírica para d_M ≤ 1:   39.98%
Valor teórico correspondiente:      39.35%
Cobertura empírica de la región 95 %: 94.32%


## 11. Relación con los filtros de Kalman

En un modelo lineal estocástico

$$
x_{k+1}=F_kx_k+G_kw_k,
\qquad
z_k=H_kx_k+v_k,
$$

se suelen especificar

$$w_k\sim\mathcal N(0,Q_k),\qquad v_k\sim\mathcal N(0,R_k).$$

Si $Q_k$ o $R_k$ no son diagonales, las componentes del ruido están correlacionadas. Para simularlas correctamente no basta multiplicar cada componente por su desviación estándar: se necesita una raíz matricial, por ejemplo

```python
L_Q = np.linalg.cholesky(Q)
w_k = L_Q @ rng.standard_normal(Q.shape[0])
```

La misma geometría aparece al representar la incertidumbre del estado estimado: la matriz $P_k$ determina las direcciones y longitudes de la elipse de covarianza.

## 12. Limitaciones y buenas prácticas

- **Cholesky:** es rápido y estable para matrices simétricas definidas positivas. Falla si hay autovalores nulos o negativos.
- **$LDL^\top$:** puede ser útil con matrices semidefinidas y evita raíces cuadradas durante la factorización. No obstante, la función didáctica anterior no incorpora pivoteo y puede fallar ante un pivote nulo. Para trabajo general conviene usar `scipy.linalg.ldl`.
- **Covarianza semidefinida:** representa una distribución degenerada sobre una recta o subespacio. Puede generarse mediante una descomposición espectral, conservando los autovalores no negativos.
- **Matriz indefinida:** no es una covarianza válida. Si aparecen autovalores negativos apreciables, debe revisarse el modelo en lugar de aplicar `sqrt(abs(D))`.
- **Tolerancias numéricas:** autovalores diminutamente negativos pueden aparecer por redondeo; cualquier corrección debe estar justificada y documentada.
- **Reproducibilidad:** una semilla fija ayuda a depurar y comparar métodos. Para Monte Carlo serio deben repetirse experimentos con múltiples semillas.

## 13. Ejercicios propuestos

1. Cambia la covarianza fuera de la diagonal de $0.5$ a $0$, $0.9$ y $-0.8$. Predice la orientación antes de ejecutar.
2. Repite el experimento con 100, 1 000 y 100 000 muestras. Compara el error de la media y de la covarianza.
3. Usa exactamente la misma matriz `x_chol` en ambos métodos y verifica si `y_chol` y `y_ldl` coinciden en este ejemplo.
4. Prueba $\Sigma=\begin{bmatrix}1&1\\1&1\end{bmatrix}$. Explica por qué Cholesky falla y por qué la distribución queda sobre una recta.
5. Construye una elipse del 99 % usando $n_{std}=\sqrt{-2\log(1-0.99)}$ y comprueba empíricamente su cobertura.

## Conclusión

Cholesky y $LDL^\top$ no "crean" correlación de forma arbitraria: construyen una transformación lineal cuya geometría reproduce exactamente la covarianza deseada en la población. Las diferencias observadas en media y covarianza muestrales se deben al tamaño finito del experimento. Esta idea es fundamental para simular perturbaciones correlacionadas y para interpretar matrices de incertidumbre en filtros de Kalman.
